# 📖 COMPLETELY FOOLPROOF GUIDE (v2.7)

### 1. Where do I look for feedback?
- **DO NOT look at the Jupyter Notebook for real-time updates.** Jupyter output is often buffered and slow.
- **LOOK AT THE BOTTOM OF THE NAPARI WINDOW.** There is a gray bar at the very bottom (the Status Bar). 
- **LOOK FOR POPUPS**: I have added "Notifications" that will appear in the top-right of the Napari window when you click or label.

### 2. How to Label (The Precise Steps):
1. **Click the Napari Window**: Ensure the window titled **'Spine Annotator'** is the front-most window on your screen.
2. **Select the Layer**: In the bottom-left list of Napari, click the layer named **'Mesh Segments'**. It must have a blue highlight around it.
3. **Point & Click**: Move your cursor to a spine and **Click it once**. 
    - **Result**: You should see a notification popup in Napari saying `SELECTED: Segment 1234`.
4. **Press a Key**: While you are hovering/selected, press **`1`**.
    - **Result**: The segment will turn **SOLID RED** immediately.

### 3. The Keys:
- **`1`**: **Spine** (Red)
- **`2`**: **Dendrite** (Green)
- **`3`**: **Soma** (Blue)
- **`0`**: **Clear**

### 4. Saving:
Go back to the Jupyter Notebook **ONLY AFTER** you are done annotating in Napari, and run the **Save Results** cell.

In [1]:
import os
import numpy as np
import pandas as pd
import napari
from napari.utils.notifications import show_info
from caveclient import CAVEclient
from meshparty import trimesh_io
from meshmash import condensed_hks_pipeline
from napari.utils.colormaps import Colormap

os.environ["QT_API"] = "pyside6"
%gui qt

dataset_name = 'minnie65_public'
materialization = 1300
CACHE_DIR = 'meshes'
OUTPUT_CSV = 'mesh_segment_labels.csv'

client = CAVEclient(dataset_name)
client.version = materialization

mm = trimesh_io.MeshMeta(
    cv_path=client.info.segmentation_source(),
    disk_cache_path=CACHE_DIR
)

In [2]:
target_id = 864691135724233643

print(f"Downloading mesh for {target_id}...")
mesh = mm.mesh(seg_id=target_id)
mesh_tuple = (mesh.vertices.astype(np.float32), mesh.faces)

print(f"Running HKS Pipeline...")
res = condensed_hks_pipeline(
    mesh_tuple, 
    simplify_target_reduction=0.7, 
    distance_threshold=3.0,
    verbose=False
)

vertices, faces = res.simple_mesh
segment_labels = res.simple_labels 
hks_comp_0 = res.condensed_features.iloc[:, 0].values
vertex_hks = hks_comp_0[segment_labels]

print(f"Mesh processed into {len(np.unique(segment_labels))} segments.")

Running HKS Pipeline...
Mesh processed into 2049 segments.


In [3]:
viewer = napari.Viewer(title=f"Spine Annotator - {target_id}")
viewer.dims.ndisplay = 3

offset = vertices.mean(axis=0)
local_vertices = (vertices - offset).astype(np.float32)

# 1. Visual Guide (Heatmap)
viewer.add_surface(
    (local_vertices, faces, vertex_hks),
    name='HKS Heatmap',
    colormap='magma', 
    opacity=0.3
)

# 2. Interaction Layer
segment_layer = viewer.add_surface(
    (local_vertices, faces, segment_labels.astype(np.float32)),
    name='Mesh Segments',
    colormap='turbo', 
    opacity=0.15
)

# 3. Annotation Layer
annotation_vertex_data = np.zeros(len(vertices), dtype=np.float32)
label_colors = [
    [0, 0, 0, 0], # 0: Trans
    [1, 0, 0, 1], # 1: Spine (Red)
    [0, 1, 0, 1], # 2: Dendrite (Green)
    [0, 0, 1, 1]  # 3: Soma (Blue)
]
discrete_cmap = Colormap(colors=label_colors, interpolation='zero')

annotation_layer = viewer.add_surface(
    (local_vertices, faces, annotation_vertex_data),
    name='YOUR ANNOTATIONS',
    colormap=discrete_cmap,
    contrast_limits=[0, 3],
    opacity=1.0,
    shading='flat'
)

annotations = {}
if os.path.exists(OUTPUT_CSV):
    existing_df = pd.read_csv(OUTPUT_CSV)
    mask = existing_df['root_id'] == target_id
    for _, row in existing_df[mask].iterrows():
        sid = int(row['segment_id'])
        annotations[sid] = row['label']
        val = {'spine': 1, 'dendrite': 2, 'soma': 3}.get(row['label'], 0)
        annotation_vertex_data[segment_labels == sid] = val

current_segment = None

def pick_id(pos, view_dir, dims):
    res = segment_layer.get_value(pos, view_dir, dims, world=True)
    val = res[0] if isinstance(res, tuple) else res
    return int(val) if val is not None else None

@viewer.mouse_move_callbacks.append
def on_move(v, event):
    global current_segment
    sid = pick_id(event.position, event.view_direction, event.dims_displayed)
    if sid is not None:
        current_segment = sid
        lbl = annotations.get(sid, "unlabeled")
        v.status = f"[SEGMENT: {sid}]   [LABEL: {lbl}]   (Press 1 to label Spine)"
    else:
        v.status = "Ready. Click or hover on the neuron surface."

@segment_layer.mouse_drag_callbacks.append
def on_click(layer, event):
    if event.type == 'mouse_press':
        sid = pick_id(event.position, event.view_direction, event.dims_displayed)
        if sid is not None:
            global current_segment
            current_segment = sid
            show_info(f"SELECTED: Segment {sid}")

def apply(l_str):
    if current_segment is not None:
        val = {'spine': 1, 'dendrite': 2, 'soma': 3}.get(l_str, 0)
        if l_str is None: annotations.pop(current_segment, None)
        else: annotations[current_segment] = l_str
        
        annotation_vertex_data[segment_labels == current_segment] = val
        annotation_layer.data = (local_vertices, faces, annotation_vertex_data)
        # Show feedback INSIDE Napari window
        show_info(f"Labled {current_segment} as {l_str or 'Cleared'}")
        print(f"Segment {current_segment} -> {l_str or 'Cleared'}", flush=True)
    else:
        show_info("ERROR: No segment selected. Click the neuron first!")

@viewer.bind_key('1', overwrite=True)
def k1(v): apply('spine')
@viewer.bind_key('2', overwrite=True)
def k2(v): apply('dendrite')
@viewer.bind_key('3', overwrite=True)
def k3(v): apply('soma')
@viewer.bind_key('0', overwrite=True)
def k0(v): apply(None)

print("Napari ready. Follow the guide in the notebook ABOVE.")

Napari ready. Follow the guide in the notebook ABOVE.


In [4]:
def save_results():
    if not annotations: return
    clean = {k: v for k, v in annotations.items() if v}
    df = pd.DataFrame([{'root_id': target_id, 'segment_id': k, 'label': v} for k, v in clean.items()])
    if os.path.exists(OUTPUT_CSV):
        old = pd.read_csv(OUTPUT_CSV)
        df = pd.concat([old[old['root_id'] != target_id], df], ignore_index=True)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"SUCCESS: Saved to {OUTPUT_CSV}", flush=True)

save_results()